In [3]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.standard_pdf_pipeline import StandardPdfPipeline

# PDF to HTML

In [4]:
import time

import torch

source = "../paper/JME_DL.pdf"

# Sanity check: MPS 是否真的可用
print(f"torch={torch.__version__}  mps_available={torch.backends.mps.is_available()}  mps_built={torch.backends.mps.is_built()}")

# JME PDF 是 born-digital 有 text layer，OCR 不需要；
# 圖片 enrichment 後面 VLM 階段再處理，這裡只打開 formula enrichment。
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.do_formula_enrichment = True
pipeline_options.generate_picture_images = True  # 圖片還是要進 HTML

# MPS 加速：Apple Silicon GPU；CodeFormula / layout / table model 都會走 MPS
pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=8,
    device=AcceleratorDevice.MPS,
)
print(f"accelerator: device={pipeline_options.accelerator_options.device} threads={pipeline_options.accelerator_options.num_threads}")

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=StandardPdfPipeline,
            pipeline_options=pipeline_options,
        ),
    }
)

t0 = time.perf_counter()
result = converter.convert(source)
elapsed = time.perf_counter() - t0
print(f"\nconvert() took {elapsed:.1f}s")

# 直接問 torch 看 MPS 是否真的有被用到 (有 enrichment 跑過就會有 allocated memory)
if torch.backends.mps.is_available():
    alloc_mb = torch.mps.current_allocated_memory() / 1024 / 1024
    driver_mb = torch.mps.driver_allocated_memory() / 1024 / 1024
    print(f"mps current allocated: {alloc_mb:.1f} MB | driver allocated: {driver_mb:.1f} MB")

print(result.document.export_to_markdown()[:2000])

torch=2.12.0  mps_available=True  mps_built=True
accelerator: device=AcceleratorDevice.MPS threads=8


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 3182.07it/s]
MPS is not available in the system. Fall back to 'CPU'
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 19199.54it/s]
[transformers] The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

from docling_core.types.doc import ImageRefMode

# 改寫到新資料夾，VLM 的結果 (../outputs-test) 保留不覆蓋方便對比
output_dir = Path("../outputs-test-enrich-std")
output_dir.mkdir(exist_ok=True)

referenced_html = output_dir / "JME_DL_referenced.html"
embedded_html = output_dir / "JME_DL_embedded.html"
split_page_html = output_dir / "JME_DL_split_page.html"

result.document.save_as_html(
    referenced_html,
    image_mode=ImageRefMode.REFERENCED,
)

result.document.save_as_html(
    embedded_html,
    image_mode=ImageRefMode.EMBEDDED,
)

result.document.save_as_html(
    split_page_html,
    image_mode=ImageRefMode.REFERENCED,
    split_page_view=True,
)

print(f"Saved: {referenced_html}")
print(f"Saved: {embedded_html}")
print(f"Saved: {split_page_html}")

# Sanity check: 比對原 PDF 頁數 vs HTML 產生的 page divs，以及有沒有空頁
import re
import pypdfium2 as pdfium

pdf_pages = len(pdfium.PdfDocument(source))
html = split_page_html.read_text()
indices = [m.start() for m in re.finditer(r"<div class='page'>", html)]
indices.append(html.find("</body>"))
empties = []
for i in range(len(indices) - 1):
    chunk = html[indices[i]:indices[i + 1]]
    text = re.sub(r"<[^>]+>", " ", chunk)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) == 0:
        empties.append(i + 1)

print(f"\nPDF pages:       {pdf_pages}")
print(f"HTML page divs:  {len(indices) - 1}")
print(f"Empty page divs: {empties or 'none'}")

Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with

Saved: ../outputs-test-std/JME_DL_referenced.html
Saved: ../outputs-test-std/JME_DL_embedded.html
Saved: ../outputs-test-std/JME_DL_split_page.html

PDF pages:       26
HTML page divs:  26
Empty page divs: none


# 2. Then

In [29]:
import json
from typing import Any

from pydantic import Field

from docling_core.transforms.serializer.common import create_ser_result
from docling_core.transforms.serializer.html import HTMLDocSerializer, HTMLOutputStyle, HTMLParams
from docling_core.types.doc.document import DocItem, GroupItem


def _first_page_no(item, doc):
    """DocItem 直接看 prov；GroupItem 沒 prov，遞迴到第一個有 prov 的後代。"""
    if isinstance(item, DocItem) and item.prov:
        return item.prov[0].page_no
    if isinstance(item, GroupItem):
        for child_ref in item.children:
            pg = _first_page_no(child_ref.resolve(doc), doc)
            if pg is not None:
                return pg
    return None


class AnnotatingHTMLDocSerializer(HTMLDocSerializer):
    """Subclass that:
    1. Wraps each top-level region with data-region-id / data-source-page / data-self-ref.
    2. 按 source page 分組，每頁一個 <div class='page' data-source-page='N'>。
    其他渲染（CSS、head、table、figure、formula）完全沿用 Docling 預設。"""

    region_counter: int = 0
    page_origin_map: dict = Field(default_factory=dict)
    region_index: dict = Field(default_factory=dict)

    def _serialize_body(self, **kwargs: Any):
        pages_to_regions: dict = {}
        page_order: list = []

        for child_ref in self.doc.body.children:
            item = child_ref.resolve(self.doc)
            ser_res = self.serialize(item=item, **kwargs)
            text = ser_res.text
            if not text.strip():
                continue  # captions/footnotes 已被 floating item 吸走

            page_no = _first_page_no(item, self.doc)
            anchor_id = f"region-{self.region_counter:04d}"
            self.region_counter += 1

            label = getattr(item, "label", None)
            label_str = label.value if hasattr(label, "value") else (str(label) if label else "group")

            wrapped = (
                f'<div data-region-id="{anchor_id}" '
                f'data-source-page="{page_no}" '
                f'data-self-ref="{item.self_ref}" '
                f'data-label="{label_str}">\n'
                f'{text}\n</div>'
            )

            if page_no not in pages_to_regions:
                pages_to_regions[page_no] = []
                page_order.append(page_no)
            pages_to_regions[page_no].append(wrapped)

            self.page_origin_map.setdefault(page_no, []).append(anchor_id)
            self.region_index[anchor_id] = {
                "self_ref": item.self_ref,
                "page_no": page_no,
                "label": label_str,
            }

        # 每個 source page 一個 .page 容器
        page_html_parts = []
        for page_no in page_order:
            regions_html = "\n".join(pages_to_regions[page_no])
            page_html_parts.append(
                f"<div class='page' data-source-page='{page_no}'>\n{regions_html}\n</div>"
            )
        body_inner = "\n".join(page_html_parts)

        # 用 Docling 原本的 head / CSS（split_page 的卡片樣式），
        # 但 split_page 是靠 <table> 居中 .page，我們沒有 table，所以加一條 override 讓 .page 水平居中
        head = self._generate_head()
        center_css = "<style>body { text-align: center; } .page { margin-left: auto; margin-right: auto; text-align: left; }</style>"
        head = head.replace("</head>", f"{center_css}\n</head>")
        full_html = f"<!DOCTYPE html>\n<html>\n{head}\n<body>\n{body_inner}\n</body>\n</html>"
        return create_ser_result(text=full_html, span_source=[])


# REFERENCED 模式下 HTMLDocSerializer.serialize() 不會主動寫圖檔到磁碟，
# 必須先用 _with_pictures_refs 把 PictureItem 的 PNG 落地到 _artifacts/，
# 並把 item.image.uri 改成相對於 reference_path 的路徑；之後 serializer 才會
# 吐出 <figure><img src="JME_DL_annotated_artifacts/image_xxx.png">。
annotated_artifacts_dir = output_dir / "JME_DL_annotated_artifacts"
doc_with_refs = result.document._with_pictures_refs(
    image_dir=annotated_artifacts_dir,
    page_no=None,
    reference_path=output_dir,
)

serializer = AnnotatingHTMLDocSerializer(
    doc=doc_with_refs,
    params=HTMLParams(image_mode=ImageRefMode.REFERENCED, output_style=HTMLOutputStyle.SPLIT_PAGE),
)
annotated_html = serializer.serialize().text
page_origin_map = serializer.page_origin_map
region_index = serializer.region_index

annotated_html_path = output_dir / "JME_DL_annotated.html"
page_map_path = output_dir / "JME_DL_page_origin_map.json"
region_index_path = output_dir / "JME_DL_region_index.json"

annotated_html_path.write_text(annotated_html)
page_map_path.write_text(json.dumps(page_origin_map, indent=2, ensure_ascii=False))
region_index_path.write_text(json.dumps(region_index, indent=2, ensure_ascii=False))

print(f"Saved: {annotated_html_path}")
print(f"Saved: {page_map_path}")
print(f"Saved: {region_index_path}")
print(f"Artifacts dir:   {annotated_artifacts_dir} ({len(list(annotated_artifacts_dir.glob('*.png')))} images)")
print(f"Total regions:   {len(region_index)}")
print(f"Pages covered:   {sorted(p for p in page_origin_map if p is not None)}")
print(f"Pages w/o prov:  {page_origin_map.get(None, [])}")


Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with MathML
Could not parse formula with

Saved: ../outputs-test-std/JME_DL_annotated.html
Saved: ../outputs-test-std/JME_DL_page_origin_map.json
Saved: ../outputs-test-std/JME_DL_region_index.json
Artifacts dir:   ../outputs-test-std/JME_DL_annotated_artifacts (18 images)
Total regions:   468
Pages covered:   [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
Pages w/o prov:  []


In [30]:
import base64
from io import BytesIO

import pypdfium2 as pdfium
from bs4 import BeautifulSoup

# 用 pypdfium2 重新 render PDF 每一頁 → base64 PNG
# (不依賴 Docling 的 generate_page_images，這樣不必重跑 docling pipeline)
pdf = pdfium.PdfDocument(source)
page_img_b64: dict[int, str] = {}
for i in range(len(pdf)):
    page_no = i + 1  # Docling 用 1-based page number
    pil_img = pdf[i].render(scale=2.0).to_pil()  # scale=2 ≈ 144dpi，debug 看夠清楚
    buf = BytesIO()
    pil_img.save(buf, format="PNG", optimize=True)
    page_img_b64[page_no] = base64.b64encode(buf.getvalue()).decode("ascii")

# 把每個 <div class="page" data-source-page="N"> 改成兩欄 table row
soup = BeautifulSoup(annotated_html, "html.parser")
body = soup.body

# 額外的 debug CSS：兩欄 layout、PDF 圖 sticky 跟著捲動方便對比
debug_style = soup.new_tag("style")
debug_style.string = """
.debug-table { border-collapse: collapse; width: 100%; }
.debug-table td { vertical-align: top; padding: 8px; border-bottom: 1px solid #ddd; }
.debug-table td.pdf-col { width: 45%; background: #f5f5f5; }
.debug-table td.pdf-col img { width: 100%; height: auto; border: 1px solid #999;
                              position: sticky; top: 10px; }
.debug-table td.html-col { width: 55%; }
.debug-table td.html-col .page { border: none; box-shadow: none; padding: 0;
                                  max-width: none; margin: 0; }
[data-region-id] { outline: 1px dashed transparent; }
[data-region-id]:hover { outline-color: #c33; background: #fff8e0; }
[data-region-id]::before {
    content: attr(data-label) " · p" attr(data-source-page) " · " attr(data-region-id);
    display: block; font-size: 10px; color: #888; font-family: monospace;
    margin-bottom: 2px;
}
"""
soup.head.append(debug_style)

# 抓出所有 .page divs，按出現順序組 table
page_divs = body.find_all("div", class_="page")
table = soup.new_tag("table", **{"class": "debug-table"})
tbody = soup.new_tag("tbody")
table.append(tbody)

for page_div in page_divs:
    page_no = int(page_div.get("data-source-page"))
    tr = soup.new_tag("tr")

    td_pdf = soup.new_tag("td", **{"class": "pdf-col"})
    img = soup.new_tag(
        "img",
        src=f"data:image/png;base64,{page_img_b64[page_no]}",
        alt=f"PDF page {page_no}",
    )
    td_pdf.append(img)
    tr.append(td_pdf)

    td_html = soup.new_tag("td", **{"class": "html-col"})
    # 把 page_div 從 body 抽出移進 td
    page_div.extract()
    td_html.append(page_div)
    tr.append(td_html)

    tbody.append(tr)

body.clear()
body.append(table)

debug_html_path = output_dir / "JME_DL_annotated_debug.html"
debug_html_path.write_text(str(soup))
print(f"Saved: {debug_html_path}")
print(f"Pages embedded: {len(page_img_b64)}")
print(f"File size: {debug_html_path.stat().st_size / 1024:.1f} KB")


Saved: ../outputs-test-std/JME_DL_annotated_debug.html
Pages embedded: 26
File size: 14927.3 KB
